---
## Install Required Libraries

In [2]:
!pip install transformers datasets seqeval evaluate accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


---
## Dataset Selection


In [3]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [4]:
import nltk
from datasets import Dataset, DatasetDict, Features, Sequence, Value, ClassLabel

nltk.download('conll2000',   quiet=True)
nltk.download('treebank',    quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.corpus import conll2000, treebank

raw_sentences = []
for sent in conll2000.iob_sents():
    tokens     = [w          for w, pos, chunk in sent]
    pos_seq    = [pos        for w, pos, chunk in sent]
    chunk_seq  = [chunk      for w, pos, chunk in sent]
    raw_sentences.append((tokens, pos_seq, chunk_seq))

print(f"Total sentences loaded: {len(raw_sentences)}")
print("Sample:", raw_sentences[0])

all_pos_tags   = sorted({tag  for _, pos, _   in raw_sentences for tag  in pos})
all_chunk_tags = sorted({tag  for _, _,   chk in raw_sentences for tag  in chk})

pos_label2id   = {t: i for i, t in enumerate(all_pos_tags)}
chunk_label2id = {t: i for i, t in enumerate(all_chunk_tags)}

# Convert to integer IDs
records = [
    {
        "tokens"     : tokens,
        "pos_tags"   : [pos_label2id[t]   for t in pos_seq],
        "chunk_tags" : [chunk_label2id[t] for t in chunk_seq],
    }
    for tokens, pos_seq, chunk_seq in raw_sentences
]

#  Train / validation / test split
import random
random.seed(42)
random.shuffle(records)
n = len(records)
train_recs = records[:int(0.8*n)]
val_recs   = records[int(0.8*n):int(0.9*n)]
test_recs  = records[int(0.9*n):]

# Build HuggingFace DatasetDict
def make_hf_dataset(recs):
    return Dataset.from_dict({
        "tokens"     : [r["tokens"]      for r in recs],
        "pos_tags"   : [r["pos_tags"]    for r in recs],
        "chunk_tags" : [r["chunk_tags"]  for r in recs],
    })

dataset = DatasetDict({
    "train"     : make_hf_dataset(train_recs),
    "validation": make_hf_dataset(val_recs),
    "test"      : make_hf_dataset(test_recs),
})

print("\nDataset structure:")
print(dataset)
print("\nFeatures:", dataset['train'].features)

Total sentences loaded: 10948
Sample: (['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NN', 'NNS', 'IN', 'NNP', ',', 'JJ', 'IN', 'NN', 'NN', ',', 'VB', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NNP', 'CC', 'NNP', 'POS', 'JJ', 'NNS', '.'], ['B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'I-VP', 'I-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-SBAR', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADJP', 'B-PP', 'B-NP', 'B-NP', 'O', 'B-VP', 'I-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'I-NP', 'O'])

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],

In [5]:
#  Inspect a Sample & Confirm Label Names
sample = dataset['train'][0]

print("Sample tokens   :", sample['tokens'])
print("POS tag IDs     :", sample['pos_tags'])
print("Chunk tag IDs   :", sample['chunk_tags'])

# Label name lists come directly from the vocabs we built during dataset creation
pos_label_names   = all_pos_tags
chunk_label_names = all_chunk_tags

print("\n--- POS Tag Labels ---")
print(pos_label_names)
print("\n--- Chunk Tag Labels ---")
print(chunk_label_names)
print(f"\nTotal POS labels  : {len(pos_label_names)}")
print(f"Total Chunk labels: {len(chunk_label_names)}")

Sample tokens   : ['No', 'replacement', 'was', 'immediately', 'named', '.']
POS tag IDs     : [10, 18, 34, 26, 36, 6]
Chunk tag IDs   : [5, 16, 10, 21, 21, 22]

--- POS Tag Labels ---
['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']

--- Chunk Tag Labels ---
['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']

Total POS labels  : 44
Total Chunk labels: 23


---
## Data Preprocessing


In [6]:
# Model Checkpoint
MODEL_CHECKPOINT = "distilbert-base-uncased"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Tokenizer loaded: {MODEL_CHECKPOINT}")

sample_words = ["John", "works", "at", "Google", "in", "California"]
encoded = tokenizer(sample_words, is_split_into_words=True)
print("\nWord IDs (tracks which word each subtoken belongs to):")
print(encoded.word_ids())

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased

Word IDs (tracks which word each subtoken belongs to):
[None, 0, 1, 2, 3, 4, 5, None]


In [7]:
# Label Alignment Function
def tokenize_and_align_labels(examples, label_column):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []

    for i, labels in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


print("✅ Label alignment function defined.")

✅ Label alignment function defined.


In [8]:
# Preprocess for POS Tagging
pos_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("✅ POS tagging dataset tokenized and labels aligned.")
print("\nSample preprocessed record keys:", list(pos_tokenized['train'][0].keys()))
print("input_ids (first 10)  :", pos_tokenized['train'][0]['input_ids'][:10])
print("attention_mask (first 10):", pos_tokenized['train'][0]['attention_mask'][:10])
print("labels (first 10)     :", pos_tokenized['train'][0]['labels'][:10])
print("  (Note: -100 = special/subword token, ignored in loss)")

Map:   0%|          | 0/8758 [00:00<?, ? examples/s]

Map:   0%|          | 0/1095 [00:00<?, ? examples/s]

Map:   0%|          | 0/1095 [00:00<?, ? examples/s]

✅ POS tagging dataset tokenized and labels aligned.

Sample preprocessed record keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids (first 10)  : [101, 2053, 6110, 2001, 3202, 2315, 1012, 102]
attention_mask (first 10): [1, 1, 1, 1, 1, 1, 1, 1]
labels (first 10)     : [-100, 10, 18, 34, 26, 36, 6, -100]
  (Note: -100 = special/subword token, ignored in loss)


In [9]:
#  Preprocess for Chunking
chunk_tokenized = dataset.map(
    lambda x: tokenize_and_align_labels(x, "chunk_tags"),
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("✅ Chunking dataset tokenized and labels aligned.")
print("input_ids (first 10)  :", chunk_tokenized['train'][0]['input_ids'][:10])
print("labels (first 10)     :", chunk_tokenized['train'][0]['labels'][:10])

Map:   0%|          | 0/8758 [00:00<?, ? examples/s]

Map:   0%|          | 0/1095 [00:00<?, ? examples/s]

Map:   0%|          | 0/1095 [00:00<?, ? examples/s]

✅ Chunking dataset tokenized and labels aligned.
input_ids (first 10)  : [101, 2053, 6110, 2001, 3202, 2315, 1012, 102]
labels (first 10)     : [-100, 5, 16, 10, 21, 21, 22, -100]


---
##Model Setup



In [10]:
# POS label mappings
pos_id2label = {i: label for i, label in enumerate(pos_label_names)}
pos_label2id = {label: i for i, label in enumerate(pos_label_names)}

# Chunk label mappings
chunk_id2label = {i: label for i, label in enumerate(chunk_label_names)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_names)}

print("POS id2label:", pos_id2label)
print("\nChunk id2label:", chunk_id2label)

POS id2label: {0: '#', 1: '$', 2: "''", 3: '(', 4: ')', 5: ',', 6: '.', 7: ':', 8: 'CC', 9: 'CD', 10: 'DT', 11: 'EX', 12: 'FW', 13: 'IN', 14: 'JJ', 15: 'JJR', 16: 'JJS', 17: 'MD', 18: 'NN', 19: 'NNP', 20: 'NNPS', 21: 'NNS', 22: 'PDT', 23: 'POS', 24: 'PRP', 25: 'PRP$', 26: 'RB', 27: 'RBR', 28: 'RBS', 29: 'RP', 30: 'SYM', 31: 'TO', 32: 'UH', 33: 'VB', 34: 'VBD', 35: 'VBG', 36: 'VBN', 37: 'VBP', 38: 'VBZ', 39: 'WDT', 40: 'WP', 41: 'WP$', 42: 'WRB', 43: '``'}

Chunk id2label: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-CONJP', 3: 'B-INTJ', 4: 'B-LST', 5: 'B-NP', 6: 'B-PP', 7: 'B-PRT', 8: 'B-SBAR', 9: 'B-UCP', 10: 'B-VP', 11: 'I-ADJP', 12: 'I-ADVP', 13: 'I-CONJP', 14: 'I-INTJ', 15: 'I-LST', 16: 'I-NP', 17: 'I-PP', 18: 'I-PRT', 19: 'I-SBAR', 20: 'I-UCP', 21: 'I-VP', 22: 'O'}


In [11]:
# Initialize POS Tagging Model
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_names),  # number of POS classes
    id2label=pos_id2label,
    label2id=pos_label2id,
    ignore_mismatched_sizes=True
)

print(f"✅ POS model loaded.")
print(f"   Backbone     : {MODEL_CHECKPOINT}")
print(f"   Num POS labels: {len(pos_label_names)}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ POS model loaded.
   Backbone     : distilbert-base-uncased
   Num POS labels: 44


In [12]:
# Initialize Chunking Model
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id,
    ignore_mismatched_sizes=True
)

print(f"✅ Chunking model loaded.")
print(f"   Backbone        : {MODEL_CHECKPOINT}")
print(f"   Num Chunk labels: {len(chunk_label_names)}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Chunking model loaded.
   Backbone        : distilbert-base-uncased
   Num Chunk labels: 23


---
##Training



In [13]:
# Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print("✅ Data collator created (dynamic padding enabled).")

✅ Data collator created (dynamic padding enabled).


In [14]:
# seqeval Metric
seqeval_metric = evaluate.load("seqeval")

print("✅ seqeval metric loaded.")

✅ seqeval metric loaded.


In [15]:
# Compute Metrics Factory
def make_compute_metrics(label_names):

    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions = np.argmax(logits, axis=-1)

        true_labels, true_predictions = [], []

        for pred_seq, label_seq in zip(predictions, labels):
            true_label_row, true_pred_row = [], []
            for pred_id, label_id in zip(pred_seq, label_seq):
                if label_id != -100:
                    true_label_row.append(label_names[label_id])
                    true_pred_row.append(label_names[pred_id])
            true_labels.append(true_label_row)
            true_predictions.append(true_pred_row)

        # Compute seqeval metrics
        results = seqeval_metric.compute(
            predictions=true_predictions,
            references=true_labels
        )

        return {
            "precision": results["overall_precision"],
            "recall"   : results["overall_recall"],
            "f1"       : results["overall_f1"],
            "accuracy" : results["overall_accuracy"],
        }

    return compute_metrics


print("✅ compute_metrics factory defined.")

✅ compute_metrics factory defined.


In [16]:
#  Training Arguments (shared for both models)
def make_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_steps=100,
        report_to="none",
        push_to_hub=False
    )

print("✅ Training arguments defined.")

✅ Training arguments defined.


In [17]:
# Train POS Tagging Model
print("🚀 Starting POS Tagging training...\n")

pos_trainer = Trainer(
    model=pos_model,
    args=make_training_args("./pos_model"),
    train_dataset=pos_tokenized["train"],
    eval_dataset=pos_tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_names)
)

pos_train_result = pos_trainer.train()
print("\n✅ POS Tagging training complete!")
print(pos_train_result)

🚀 Starting POS Tagging training...



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.151138,0.129206,0.952683,0.955115,0.953898,0.966616
2,0.098359,0.100974,0.961062,0.961196,0.961129,0.972444
3,0.076997,0.096634,0.962625,0.963563,0.963094,0.973834


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ POS Tagging training complete!
TrainOutput(global_step=1644, training_loss=0.24441446048499893, metrics={'train_runtime': 172.9633, 'train_samples_per_second': 151.905, 'train_steps_per_second': 9.505, 'total_flos': 383975637728928.0, 'train_loss': 0.24441446048499893, 'epoch': 3.0})


In [18]:
# Train Chunking Model
print("🚀 Starting Chunking training...\n")

chunk_trainer = Trainer(
    model=chunk_model,
    args=make_training_args("./chunk_model"),
    train_dataset=chunk_tokenized["train"],
    eval_dataset=chunk_tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_names)
)

chunk_train_result = chunk_trainer.train()
print("\n✅ Chunking training complete!")
print(chunk_train_result)

🚀 Starting Chunking training...



Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.135948,0.112950,0.951210,0.959750,0.955461,0.972328
2,0.091403,0.097364,0.958528,0.965457,0.961980,0.976535
3,0.076606,0.095024,0.960366,0.966066,0.963207,0.976574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Chunking training complete!
TrainOutput(global_step=1644, training_loss=0.16576879288448323, metrics={'train_runtime': 193.7004, 'train_samples_per_second': 135.642, 'train_steps_per_second': 8.487, 'total_flos': 383829950623560.0, 'train_loss': 0.16576879288448323, 'epoch': 3.0})


---
##  Evaluation



In [19]:
# Evaluate POS Model on Test Set
print("📊 Evaluating POS Tagging model on test set...")
pos_eval_results = pos_trainer.evaluate(eval_dataset=pos_tokenized["test"])

print("\n=== POS Tagging — Test Set Results ===")
print(f"  Precision : {pos_eval_results['eval_precision']:.4f}")
print(f"  Recall    : {pos_eval_results['eval_recall']:.4f}")
print(f"  F1 Score  : {pos_eval_results['eval_f1']:.4f}")
print(f"  Accuracy  : {pos_eval_results['eval_accuracy']:.4f}")

📊 Evaluating POS Tagging model on test set...



=== POS Tagging — Test Set Results ===
  Precision : 0.9669
  Recall    : 0.9670
  F1 Score  : 0.9670
  Accuracy  : 0.9761


In [20]:
# Evaluate Chunking Model on Test Set
print("📊 Evaluating Chunking model on test set...")
chunk_eval_results = chunk_trainer.evaluate(eval_dataset=chunk_tokenized["test"])

print("\n=== Chunking — Test Set Results ===")
print(f"  Precision : {chunk_eval_results['eval_precision']:.4f}")
print(f"  Recall    : {chunk_eval_results['eval_recall']:.4f}")
print(f"  F1 Score  : {chunk_eval_results['eval_f1']:.4f}")
print(f"  Accuracy  : {chunk_eval_results['eval_accuracy']:.4f}")

📊 Evaluating Chunking model on test set...



=== Chunking — Test Set Results ===
  Precision : 0.9592
  Recall    : 0.9637
  F1 Score  : 0.9614
  Accuracy  : 0.9758


---
## Inference



In [21]:
import torch

def predict_tags(sentence, model, tokenizer, id2label):
    words = sentence.split()
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    word_ids = encoding.word_ids()

    model.eval()
    with torch.no_grad():
        outputs = model(**encoding)

    # Convert logits to label IDs
    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

    results = []
    seen_words = set()

    for word_idx, pred_id in zip(word_ids, predictions):
        if word_idx is None or word_idx in seen_words:
            continue
        seen_words.add(word_idx)
        results.append((words[word_idx], id2label[pred_id]))

    return results


print("✅ Inference function defined.")

✅ Inference function defined.


In [23]:
import torch

def predict_tags(sentence, model, tokenizer, id2label):
    words = sentence.split()
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    # Move inputs to same device as model
    device = next(model.parameters()).device
    encoding = {k: v.to(device) for k, v in encoding.items()}

    word_ids = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True
    ).word_ids()

    model.eval()
    with torch.no_grad():
        outputs = model(**encoding)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()

    results = []
    seen_words = set()
    for word_idx, pred_id in zip(word_ids, predictions):
        if word_idx is None or word_idx in seen_words:
            continue
        seen_words.add(word_idx)
        results.append((words[word_idx], id2label[pred_id]))

    return results


# Run Inference on Example Sentences
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Microsoft announced a new product yesterday"
]

for sentence in test_sentences:
    print(f"\nInput: {sentence}")

    pos_preds = predict_tags(sentence, pos_model, tokenizer, pos_id2label)
    print(f"\n  {'Word':<18} {'POS Tag'}")
    print("  " + "-" * 30)
    for word, tag in pos_preds:
        print(f"  {word:<18} {tag}")

    chunk_preds = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)
    print(f"\n  {'Word':<18} {'Chunk Tag'}")
    print("  " + "-" * 30)
    for word, tag in chunk_preds:
        print(f"  {word:<18} {tag}")

    print("\n" + "=" * 55)


Input: John works at Google in California

  Word               POS Tag
  ------------------------------
  John               NNP
  works              VBZ
  at                 IN
  Google             NNP
  in                 IN
  California         NNP

  Word               Chunk Tag
  ------------------------------
  John               B-NP
  works              B-VP
  at                 B-PP
  Google             B-NP
  in                 B-PP
  California         B-NP


Input: The quick brown fox jumps over the lazy dog

  Word               POS Tag
  ------------------------------
  The                DT
  quick              NNP
  brown              NNP
  fox                NNP
  jumps              VBZ
  over               IN
  the                DT
  lazy               JJ
  dog                NN

  Word               Chunk Tag
  ------------------------------
  The                B-NP
  quick              I-NP
  brown              I-NP
  fox                I-NP
  jumps             

---
## Comparison — POS Tagging vs Chunking



In [25]:
# Side-by-Side Comparison Table
print("       MODEL PERFORMANCE COMPARISON")
print("=" * 55)
print(f"{'Metric':<15} {'POS Tagging':>15} {'Chunking':>15}")
print("-" * 55)
metrics = ["precision", "recall", "f1", "accuracy"]
for m in metrics:
    pos_val   = pos_eval_results.get(f"eval_{m}", 0)
    chunk_val = chunk_eval_results.get(f"eval_{m}", 0)
    print(f"{m.capitalize():<15} {pos_val:>15.4f} {chunk_val:>15.4f}")
print("=" * 55)
print("\n📌 Observation: POS Tagging typically achieves higher F1 than")
print("   Chunking because it predicts flat per-token labels, while")
print("   Chunking requires coherent BIO span predictions.")

       MODEL PERFORMANCE COMPARISON
Metric              POS Tagging        Chunking
-------------------------------------------------------
Precision                0.9669          0.9592
Recall                   0.9670          0.9637
F1                       0.9670          0.9614
Accuracy                 0.9761          0.9758

📌 Observation: POS Tagging typically achieves higher F1 than
   Chunking because it predicts flat per-token labels, while
   Chunking requires coherent BIO span predictions.


---
## Report / Blog



In [26]:
# Save Both Models
pos_trainer.save_model("./pos_model_final")
chunk_trainer.save_model("./chunk_model_final")
tokenizer.save_pretrained("./pos_model_final")
tokenizer.save_pretrained("./chunk_model_final")

print("✅ Both models saved!")
print("   POS model   → ./pos_model_final")
print("   Chunk model → ./chunk_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Both models saved!
   POS model   → ./pos_model_final
   Chunk model → ./chunk_model_final
